# Data modeling 7: loading and late data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-07-loading-and-late-data.ipynb)

A model is only as correct as the process that loads it. Loads run every day, fail halfway, get retried, and
receive data late, out of order and more than once. This lesson covers the loading patterns that keep a
dimensional model correct under those conditions: incremental loads with watermarks, idempotent writes,
change data capture, updating accumulating snapshots, late-arriving facts, early-arriving facts, and building
periodic snapshots from transactions.

**Prerequisites:** [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb), [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) and [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb).

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order) and `src_payments` (one row per payment; an order can have several): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_payments (payment_id varchar primary key, order_id int, paid_ts timestamp, method varchar, amount decimal(10, 2));
insert into src_payments values  -- order 1003 was paid in two parts; 1008 is unpaid and 1009 was cancelled
    ('P01', 1001, '2026-01-05 09:13', 'card',    76.99),
    ('P02', 1002, '2026-01-09 14:31', 'card',    60.00),
    ('P03', 1003, '2026-01-14 11:06', 'card',   132.50),
    ('P04', 1003, '2026-02-13 00:00', 'invoice', 100.00),
    ('P05', 1004, '2026-01-28 16:41', 'card',    34.99),
    ('P06', 1005, '2026-02-03 10:21', 'card',   114.99),
    ('P07', 1006, '2026-03-13 00:00', 'invoice', 400.00),
    ('P08', 1007, '2026-02-14 09:51', 'cash',    54.00),
    ('P09', 1010, '2026-02-27 08:31', 'cash',    33.00);
""")
con.sql("show tables").show()

## Full, incremental and idempotent loads

A **full load** rebuilds the table from all source data every run. It is simple and always correct, and the
right choice while the data is small: build the new table and swap it in atomically.

An **incremental load** processes only what changed since the last run. It keeps a **watermark**, such as the
highest `updated_at` already loaded, and reads rows above it. It is cheaper but has two risks: rows that
arrive with an older timestamp than the watermark are skipped, and a retried run may process the same rows
twice.

So every load must be **idempotent**: running it again with the same input leaves the table unchanged. Three
write patterns, from worst to best:

- `insert` (append): not idempotent. A retry duplicates every row it wrote.
- `merge` on the table's key (upsert): idempotent; reprocessed rows overwrite themselves.
- **delete-then-insert by partition**: delete everything for the affected dates, insert the recomputed rows,
  in one transaction. Idempotent, and handles rows that should disappear too.

**Where it matters**
- Every scheduled load. Retries, backfills and reruns after a bug fix all depend on idempotency.

**Watch out for**
- Advancing the watermark before the write commits: a failure in between skips data forever.
- Watermarks on the load time instead of a source change time: rows loaded slowly by the source are missed.

### Example

Load the order fact incrementally with plain inserts, then rerun the same batch; repeat with `merge`.

In [ ]:
con.sql("""
create table fact_order_append (order_id int, status varchar, amount decimal(10, 2));
create table fact_order_merge  (order_id int primary key, status varchar, amount decimal(10, 2));
create table etl_watermark (table_name varchar primary key, loaded_to timestamp);
insert into etl_watermark values ('fact_order', '2026-01-31 23:59:59');   -- January is already loaded
""")

batch = """select order_id, status, shipping_fee as amount from src_orders
           where order_ts > (select loaded_to from etl_watermark where table_name = 'fact_order')"""

for run in ["first run", "retry of the same batch"]:
    con.sql(f"insert into fact_order_append {batch}")
    con.sql(f"""
        merge into fact_order_merge t using ({batch}) s on t.order_id = s.order_id
        when matched then update set status = s.status, amount = s.amount
        when not matched then insert values (s.order_id, s.status, s.amount)
    """)
    rows = con.sql("select (select count(*) from fact_order_append), (select count(*) from fact_order_merge)").fetchall()[0]
    print(f"after the {run}: append table {rows[0]} rows, merge table {rows[1]} rows")

con.sql("update etl_watermark set loaded_to = (select max(order_ts) from src_orders) where table_name = 'fact_order'")
con.sql("select * from etl_watermark").show()

**What to notice:** The batch holds February's 6 orders. After the retry the append table has 12 rows, every February order twice,
while the merge table still has 6. The watermark moves to the latest order timestamp only after the writes, so a
crash before that line just repeats the batch, which the merge makes harmless.

## Change data capture and current-state tables

**Change data capture (CDC)** delivers every insert, update and delete in a source table as an event, usually
from the database's log (Debezium, AWS DMS, Fivetran). A CDC feed is ordered per key but arrives in batches
that can contain several changes to the same row, and sometimes the same change twice.

To maintain a current-state table from it:

1. Within the batch, keep only the **latest** event per key (`row_number()` over the key, ordered by the
   source's change sequence or timestamp, descending).
2. `merge` that one row per key: update on match, insert when new, and **delete** (or mark deleted) when the
   latest operation is a delete.

Skipping step 1 makes the `merge` fail (many warehouses reject a merge where one target row matches several
source rows) or, worse, apply the changes in an arbitrary order.

**Where it matters**
- Replicating operational tables into staging, and feeding Type 2 dimensions with the same events (lesson 3).

**Watch out for**
- Ordering by an arrival timestamp instead of the source's commit sequence: retries can arrive out of order.
- Hard-deleting rows that facts still reference: prefer a soft-delete flag (`is_deleted`) in dimensions.

### Example

A CDC batch for orders: order 1006 was delivered, order 1008 shipped and then delivered within the same batch,
order 1011 was created, order 1009 was deleted from the source, and one event was delivered twice.

In [ ]:
con.sql("""
create table stg_orders_current as select order_id, order_ts, status, delivered_ts, false as is_deleted from src_orders;

create table cdc_orders (lsn int, op varchar, order_id int, order_ts timestamp, status varchar, delivered_ts timestamp);
insert into cdc_orders values
    (501, 'u', 1006, '2026-02-11 13:15', 'delivered', '2026-03-02 11:00'),
    (502, 'u', 1008, '2026-02-20 18:05', 'shipped',   null),
    (503, 'u', 1008, '2026-02-20 18:05', 'delivered', '2026-03-03 09:30'),
    (503, 'u', 1008, '2026-02-20 18:05', 'delivered', '2026-03-03 09:30'),   -- the same event delivered twice
    (504, 'c', 1011, '2026-03-03 15:00', 'placed',    null),
    (505, 'd', 1009, '2026-02-24 12:00', null,        null);
""")
con.sql("""
merge into stg_orders_current t
using (
    select * from cdc_orders
    qualify row_number() over (partition by order_id order by lsn desc) = 1   -- latest event per order
) s
on t.order_id = s.order_id
when matched and s.op = 'd' then update set is_deleted = true
when matched then update set status = s.status, delivered_ts = s.delivered_ts
when not matched and s.op <> 'd' then insert values (s.order_id, s.order_ts, s.status, s.delivered_ts, false)
""")
con.sql("select * from stg_orders_current where order_id >= 1006 order by order_id").show()

**What to notice:** Order 1008 had two changes in the batch (shipped, then delivered) and one of them twice; only the latest, LSN 503,
was applied, so it is `delivered`. Order 1011 was inserted. Order 1009 is still there, marked `is_deleted`, so
any fact that references it keeps a dimension row to join to.

## Updating an accumulating snapshot

An accumulating snapshot (lesson 1) has one row per process instance, and each load fills in the milestones
that have happened since. That makes its load a `merge`: insert rows for new orders, update milestone
timestamps and lags for existing ones, and never touch rows that did not change. A milestone should only move
forward: a late or replayed event must not overwrite a newer milestone with an older value.

**Where it matters**
- Fulfilment, claims, applications, onboarding funnels: anything with a fixed sequence of steps.

**Watch out for**
- Recomputing lags in the report instead of the load: every query repeats the logic, slightly differently.

### Example

Build the fulfilment snapshot as of February 28, then merge in the state after the CDC batch above.

In [ ]:
con.sql("""
create table fact_fulfilment as
select order_id, status, order_ts, shipped_ts, delivered_ts,
       date_diff('hour', order_ts, delivered_ts) as hours_to_deliver
from src_orders;
""")
con.sql("""
merge into fact_fulfilment t
using (
    select c.order_id, c.status, c.delivered_ts, c.is_deleted, c.order_ts, o.shipped_ts
    from stg_orders_current c left join src_orders o using (order_id)
) s
on t.order_id = s.order_id
when matched and s.is_deleted then delete
when matched and s.delivered_ts is distinct from t.delivered_ts then update set
    status = s.status,
    delivered_ts = greatest(t.delivered_ts, s.delivered_ts),          -- a milestone only moves forward
    shipped_ts = coalesce(t.shipped_ts, s.delivered_ts),               -- delivered implies shipped
    hours_to_deliver = date_diff('hour', t.order_ts, greatest(t.delivered_ts, s.delivered_ts))
when not matched then insert values (s.order_id, s.status, s.order_ts, s.shipped_ts, s.delivered_ts, null)
""")
con.sql("select * from fact_fulfilment where order_id >= 1005 order by order_id").show()

**What to notice:** Orders 1006 and 1008 were updated in place, gaining a delivery time and hours to deliver (1008's shipped time
is filled from its delivery because the feed did not include it). Order 1009 was removed and 1011 added with
nulls in the milestones it has not reached. Orders 1005, 1007 and 1010 were not touched, because nothing about
them changed.

## Late-arriving facts

A fact is **late** when it arrives after facts with later event times have already been loaded: a store's
sales upload that failed for three days, a mobile event sent when the phone reconnects. Two rules keep late
facts from corrupting reports:

- Store and partition facts by **event time**, never by load time. A January sale that arrives in March belongs
  in January.
- Reprocess a **lookback window**. Each run recomputes the partitions that could have received late data (the
  last N days), with delete-then-insert, rather than only the newest day.

Also keep the load time (`_loaded_at`) as a column, so you can tell when a number changed and why last
week's report no longer matches.

**Where it matters**
- Event and mobile data, store uploads, partner files, anything with retries upstream.

**Watch out for**
- Late facts changing figures that were already published: decide and document when a period is closed.
- Resolving a late fact's dimension key by today's version instead of the version at its event time.

### Example

A daily sales summary is rebuilt each day for the last 7 days. A store upload for January 20 that should have
arrived on January 20 arrives on February 26, after the summary was built. Compare loading only the newest day
with reprocessing a lookback window that reaches back to the late data.

In [ ]:
con.sql("""
create table sales_lines as
select l.order_id, cast(o.order_ts as date) as sale_date, l.qty * l.unit_price - l.discount as net_amount,
       timestamp '2026-02-28 06:00' as _loaded_at
from src_order_lines l join src_orders o using (order_id) where o.status <> 'cancelled';

create table daily_sales as select sale_date, sum(net_amount) as net_sales from sales_lines group by sale_date;

-- the late upload: a store order from January 20, loaded on March 1
insert into sales_lines values (1099, '2026-01-20', 88.00, timestamp '2026-03-01 06:00');
""")


def rebuild_daily_sales(from_date):
    """Delete-then-insert every day from from_date: idempotent, and it picks up late rows in that window."""
    con.sql(f"""
        begin transaction;
        delete from daily_sales where sale_date >= date '{from_date}';
        insert into daily_sales
        select sale_date, sum(net_amount) from sales_lines where sale_date >= date '{from_date}' group by sale_date;
        commit;
    """)


rebuild_daily_sales("2026-02-22")  # the usual 7-day window
jan_20 = con.sql("select count(*) from daily_sales where sale_date = '2026-01-20'").fetchall()[0][0]
print(f"rows for January 20 after the 7-day rebuild: {jan_20}")
rebuild_daily_sales("2026-01-01")  # a window wide enough to reach the late row
con.sql("select * from daily_sales where sale_date between '2026-01-14' and '2026-01-28' order by sale_date").show()
con.sql("select sale_date, _loaded_at, count(*) as lines from sales_lines group by all having sale_date = '2026-01-20'").show()

**What to notice:** The normal 7-day window never looked at January 20, so the summary had no row for it although the line was in
`sales_lines`. Reprocessing from January 1 added January 20 with its 88.00. The `_loaded_at` column shows the
line arrived on March 1, which explains why January's total changed after it had been reported. The window length
is a trade-off between cost and how late your data can arrive; measure the lateness, then choose.

## Early-arriving facts and inferred members

The reverse problem: a fact arrives **before** its dimension row. An order comes in for customer 107, but the
CRM feed with customer 107 lands tomorrow. Lesson 2 mapped such facts to the `-1` unknown member, which keeps
them in totals but means updating every affected fact row once the customer arrives.

The alternative is an **inferred member**: when the fact load meets an unknown natural key, it inserts a
placeholder dimension row for that key, with a new surrogate key, the natural key, unknown attributes and a flag
`is_inferred = true`. The fact stores that surrogate key. When the real customer arrives, the dimension load
updates the placeholder **in place** (as Type 1, even for Type 2 attributes, because nothing true was recorded
yet) and clears the flag. The facts never need touching.

**Where it matters**
- Any source pair that is not synchronized: orders before CRM, events before user profiles, trades before reference data.

**Watch out for**
- Creating a new Type 2 version when the real data arrives instead of overwriting the placeholder: the early facts stay attached to a row full of unknowns.

### Example

An order arrives for unknown customer 107. Load it with an inferred member, then load the customer the next day.

In [ ]:
con.sql("""
create sequence customer_sk_seq start 1;
create table dim_customer (customer_sk int primary key, customer_id int, name varchar, segment varchar, is_inferred boolean);
insert into dim_customer select nextval('customer_sk_seq'), customer_id, name, segment, false from src_customers;
create table fact_orders (order_id int, customer_sk int, amount decimal(10, 2));
""")

# day 1: the fact load meets customer 107 and creates a placeholder before inserting the fact
con.sql("""
insert into dim_customer
select nextval('customer_sk_seq'), 107, 'Unknown (inferred)', 'Unknown', true
where not exists (select 1 from dim_customer where customer_id = 107);
insert into fact_orders select 2001, customer_sk, 75.00 from dim_customer where customer_id = 107;
""")
con.sql("select * from dim_customer where customer_id = 107").show()

# day 2: the CRM delivers customer 107; the placeholder is completed in place
con.sql("""
update dim_customer set name = 'Gamma Goods', segment = 'smb', is_inferred = false
where customer_id = 107 and is_inferred
""")
con.sql("""
select f.order_id, f.customer_sk, d.customer_id, d.name, d.segment, d.is_inferred
from fact_orders f join dim_customer d using (customer_sk)
""").show()

**What to notice:** On day 1 customer 107 got surrogate key 7 with unknown attributes and `is_inferred = true`, and order 2001 was
stored against key 7. On day 2 the same row was completed, so the order now reports as Gamma Goods, `smb`,
without any change to the fact table. With the `-1` approach, the fact row itself would have had to be found and
re-keyed.

## Building a periodic snapshot from transactions

Sources record events; many questions are about levels: how much does each customer owe at the end of each
day? A **periodic snapshot** answers it with one row per customer per day, built from the transactions:

1. Make a **spine** of every (customer, day) pair, from the date dimension and the customer dimension, so days
   with no activity still get a row.
2. Left join the day's transactions (amounts billed and paid).
3. Take a running sum per customer over the days: the balance carried forward.

Because the snapshot has a row for every day, "the balance on February 14" is a lookup, and month-end
balances are a filter, not a calculation every report repeats.

**Where it matters**
- Account balances and receivables, inventory levels, open tickets, active subscriptions.

**Watch out for**
- Skipping the spine: days without transactions have no row, and "balance on day X" returns nothing.
- Summing the snapshot over days: balances are semi-additive (lesson 1).

### Example

Build a daily receivables snapshot: amount ordered (with shipping) minus amount paid, per customer per day.

In [ ]:
con.sql("""
create table fact_receivable_daily as
with days as (select cast(d as date) as day from range(date '2026-01-01', date '2026-03-15', interval 1 day) t(d)),
spine as (select c.customer_id, d.day from src_customers c cross join days d),
billed as (
    select o.customer_id, cast(o.order_ts as date) as day, sum(l.qty * l.unit_price - l.discount) + any_value(o.shipping_fee) as amount
    from src_orders o join src_order_lines l using (order_id)
    where o.status <> 'cancelled' group by o.customer_id, o.order_id, cast(o.order_ts as date)
),
paid as (
    select o.customer_id, cast(p.paid_ts as date) as day, sum(p.amount) as amount
    from src_payments p join src_orders o using (order_id) group by all
),
daily as (
    select s.customer_id, s.day,
           coalesce((select sum(amount) from billed b where b.customer_id = s.customer_id and b.day = s.day), 0) as billed,
           coalesce((select sum(amount) from paid p   where p.customer_id = s.customer_id and p.day = s.day), 0) as paid
    from spine s
)
select customer_id, day, billed, paid,
       sum(billed - paid) over (partition by customer_id order by day) as balance
from daily
""")
con.sql("""
select customer_id, day, billed, paid, balance from fact_receivable_daily
where customer_id in (102, 103, 104) and day in ('2026-01-14', '2026-01-31', '2026-02-11', '2026-02-13', '2026-02-28', '2026-03-14')
order by customer_id, day
""").show()

In [ ]:
con.sql("""
select day as month_end, sum(balance) as total_receivable
from fact_receivable_daily
where day in ('2026-01-31', '2026-02-28')
group by day order by day
""").show()

**What to notice:** Customer 103's balance jumps to 100.00 on January 14 (an order of 232.50, of which 132.50 was paid at once) and
stays there, without any transactions, until the invoice is paid on February 13. Customer 102 owes 400.00 from
February 11 until March 13. Customer 104's 30.99 is order 1008, still unpaid. Month-end receivables are a filter on
the snapshot: 100.00 at the end of January and 430.99 at the end of February, summed across customers on one day,
never across days.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A load appends yesterday's orders. It fails after writing half of them and is retried. What happens, and how do you fix it?

**A.** The first half is written twice. Make the write idempotent: `merge` on the order key, or delete yesterday's partition and insert it again in one transaction.

**Q2.** A CDC batch contains an update and then a delete for the same order. What does your merge do if you skip the deduplication step?

**A.** Both source rows match one target row. Many engines reject the merge; others apply the events in an arbitrary order, so the order may end up updated instead of deleted.

**Q3.** Why partition a fact by event date rather than load date?

**A.** Reports filter on when things happened. Partitioning by event date puts late data in the right period, lets reprocessing target the affected partitions, and lets queries prune by the dates they ask about.

**Q4.** What is the difference between mapping an early fact to `-1` and creating an inferred member?

**A.** With `-1`, the fact must later be found and re-keyed when the customer arrives. With an inferred member, the fact points at a placeholder row for that natural key, which is completed in place, so the fact never changes.

**Q5.** Why does a daily balance snapshot need a spine of every customer and day?

**A.** Balances carry forward through days with no transactions. Without the spine those days have no row, so the balance on that day cannot be read and month-end values can be missing.

## Key takeaways

- Prefer full rebuilds while data is small; incremental loads need a watermark advanced only after the write commits.
- Every load must be idempotent: use `merge` on the key or delete-then-insert by partition, never plain appends.
- From CDC, keep the latest event per key in each batch before merging, and soft-delete rather than hard-delete.
- Accumulating snapshots are loaded with `merge`, and milestones only move forward.
- Partition facts by event time, reprocess a lookback window, and keep `_loaded_at` to explain changes.
- Early-arriving facts get an inferred dimension member that is completed in place when the real row arrives.
- Periodic snapshots come from a spine of every key and period, the period's transactions, and a running sum.

**Next:** Practise lessons 1 to 7 in the [Data modeling drills](data-modeling-drills.ipynb), then continue with [Data modeling 8: physical design and performance](data-modeling-08-physical-design-and-performance.ipynb).